In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('Data/cleaned/cleaned_data.csv')
df.drop(columns=['Name', 'Ticket'], inplace=True, errors='ignore')

# Convert categories to numbers using one-hot encoding
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Convert passenger class to ordinal numbers
df['Pclass'] = df['Pclass'].map({1: 3, 2: 2, 3: 1})

# Scale numerical columns to have a standard range
scaler = StandardScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])

# Create a feature for family size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create a feature for fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

# Create an interaction feature between age and class
df['AgeClass'] = df['Age'] * df['Pclass']

# Apply log transform to fix skewed fare data
df['LogFare'] = np.log1p(df['Fare'].abs())

# Group age into categories
df['AgeGroup'] = pd.cut(df['Age'], bins=3, labels=['Young', 'Middle', 'Old'])

# Remove highly correlated redundant columns
corr_matrix = df.corr(numeric_only=True).abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df.drop(columns=to_drop, inplace=True)

df.to_csv('Data/cleaned/final_features.csv', index=False)